# 🎬 Netflix Customer Churn & Engagement Analytics Using AI

**Objective:** Predict customer churn, understand engagement drivers, and surface actionable business recommendations using a 5,000-customer Netflix dataset.

| Section | Description |
|---------|-------------|
| 1 | Environment Setup & Data Loading |
| 2 | Data Inspection & Cleaning |
| 3 | Exploratory Data Analysis (EDA) |
| 4 | Feature Engineering |
| 5 | Churn Deep-Dive Analysis |
| 6 | Machine Learning Models |
| 7 | Model Evaluation & Comparison |
| 8 | Feature Importance |
| 9 | Key Insights & Business Recommendations |

---
## 1. Environment Setup & Data Loading

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── scikit-learn ─────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, ConfusionMatrixDisplay
)

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 13,
                     'axes.labelsize': 11, 'xtick.labelsize': 9,
                     'ytick.labelsize': 9})
NETFLIX_RED  = '#E50914'
NETFLIX_DARK = '#221F1F'
PALETTE2     = [NETFLIX_RED, '#6B6B6B']

print('Libraries loaded successfully -- OK')

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
df = pd.read_csv('netflix_customer_churn.csv')
print(f'Dataset shape : {df.shape}')
print(f'Columns       : {list(df.columns)}')
df.head()

---
## 2. Data Inspection & Cleaning

In [ ]:
# ── Basic info ────────────────────────────────────────────────────────────────
print('=== dtypes & non-null counts ===')
df.info()
print()
print('=== Missing values ===')
print(df.isnull().sum())

In [ ]:
# ── Descriptive statistics ────────────────────────────────────────────────────
df.describe(include='all').T

In [ ]:
# ── Duplicates ────────────────────────────────────────────────────────────────
dupes = df.duplicated().sum()
print(f'Duplicate rows : {dupes}')
df = df.drop_duplicates()

# ── Drop identifier column ────────────────────────────────────────────────────
df = df.drop(columns=['customer_id'])

# ── Strip whitespace from string columns ─────────────────────────────────────
str_cols = df.select_dtypes('object').columns
df[str_cols] = df[str_cols].apply(lambda c: c.str.strip())

# ── Verify churn label distribution ──────────────────────────────────────────
print('\nChurn distribution:')
print(df['churned'].value_counts())
print(f'Churn rate : {df["churned"].mean()*100:.1f}%')
print(f'\nClean dataset shape : {df.shape}')

In [ ]:
# ── Unique values in categorical columns ─────────────────────────────────────
cat_cols = df.select_dtypes('object').columns.tolist()
for col in cat_cols:
    print(f'{col:25s}: {sorted(df[col].unique())}')

---
## 3. Exploratory Data Analysis (EDA)

### 3.1 Target Variable — Churn Distribution

In [ ]:
churn_counts = df['churned'].value_counts()
labels = ['Retained', 'Churned']

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Bar
axes[0].bar(labels, churn_counts.values, color=PALETTE2, edgecolor='white', linewidth=1.2)
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 20, f'{v:,}', ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Churn Count')
axes[0].set_ylabel('Number of Customers')

# Pie
axes[1].pie(churn_counts.values, labels=labels, colors=PALETTE2,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Churn Proportion')

plt.suptitle('Customer Churn Overview', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 3.2 Numerical Feature Distributions

In [ ]:
num_cols = ['age', 'watch_hours', 'last_login_days',
            'monthly_fee', 'number_of_profiles', 'avg_watch_time_per_day']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    for churn_val, color, label in zip([0, 1], PALETTE2, ['Retained', 'Churned']):
        axes[i].hist(df[df['churned'] == churn_val][col],
                     bins=30, alpha=0.6, color=color, label=label, edgecolor='none')
    axes[i].set_title(col.replace('_', ' ').title())
    axes[i].set_ylabel('Count')
    axes[i].legend(fontsize=8)

plt.suptitle('Numerical Feature Distributions by Churn Status',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.3 Box-Plots — Churn vs Numerical Features

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(data=df, x='churned', y=col, palette=PALETTE2,
                ax=axes[i], showfliers=True, linewidth=1.2)
    axes[i].set_xticklabels(['Retained', 'Churned'])
    axes[i].set_title(col.replace('_', ' ').title())
    axes[i].set_xlabel('')

plt.suptitle('Box-Plots: Numerical Features vs Churn',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.4 Categorical Features vs Churn

In [ ]:
cat_features = ['gender', 'subscription_type', 'region',
                'device', 'payment_method', 'favorite_genre']

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    churn_rate = (df.groupby(col)['churned'].mean() * 100).sort_values(ascending=False)
    bars = axes[i].bar(churn_rate.index, churn_rate.values,
                       color=NETFLIX_RED, edgecolor='white', linewidth=0.8)
    for bar, val in zip(bars, churn_rate.values):
        axes[i].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 0.4, f'{val:.1f}%',
                     ha='center', va='bottom', fontsize=8)
    axes[i].set_title(f'Churn Rate by {col.replace("_", " ").title()}')
    axes[i].set_ylabel('Churn Rate (%)')
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].set_ylim(0, churn_rate.max() * 1.2)

plt.suptitle('Churn Rate Across Categorical Dimensions',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.5 Correlation Heatmap

In [ ]:
corr = df[num_cols + ['churned']].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Correlation Matrix (Numerical Features + Churn)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.6 Age & Watch-Hours Scatter by Churn

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for churn_val, color, label in zip([0, 1], PALETTE2, ['Retained', 'Churned']):
    sub = df[df['churned'] == churn_val]
    ax.scatter(sub['age'], sub['watch_hours'],
               alpha=0.35, s=18, c=color, label=label)

ax.set_xlabel('Age')
ax.set_ylabel('Watch Hours')
ax.set_title('Age vs Watch Hours — Churn vs Retained')
ax.legend()
plt.tight_layout()
plt.show()

### 3.7 Last Login Days vs Churn

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for churn_val, color, label in zip([0, 1], PALETTE2, ['Retained', 'Churned']):
    sns.kdeplot(df[df['churned'] == churn_val]['last_login_days'],
                ax=ax, color=color, label=label, fill=True, alpha=0.3, linewidth=2)
ax.set_xlabel('Days Since Last Login')
ax.set_ylabel('Density')
ax.set_title('Days Since Last Login — KDE by Churn Status')
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Feature Engineering

In [ ]:
df_fe = df.copy()

# ── Engagement score (higher = more engaged) ──────────────────────────────────
df_fe['engagement_score'] = (
    df_fe['watch_hours'] * 0.4
    + df_fe['avg_watch_time_per_day'] * 10
    - df_fe['last_login_days'] * 0.5
)

# ── Watch intensity flag ──────────────────────────────────────────────────────
df_fe['high_watcher'] = (df_fe['watch_hours'] >
                          df_fe['watch_hours'].median()).astype(int)

# ── Inactivity flag ───────────────────────────────────────────────────────────
df_fe['inactive_30d'] = (df_fe['last_login_days'] >= 30).astype(int)

# ── Age groups ────────────────────────────────────────────────────────────────
df_fe['age_group'] = pd.cut(df_fe['age'],
                             bins=[0, 25, 35, 45, 55, 100],
                             labels=['<25', '25-35', '35-45', '45-55', '55+'])

# ── Revenue-per-profile ───────────────────────────────────────────────────────
df_fe['fee_per_profile'] = df_fe['monthly_fee'] / df_fe['number_of_profiles'].replace(0, 1)

# ── Watch-hour segments ───────────────────────────────────────────────────────
df_fe['watch_segment'] = pd.qcut(df_fe['watch_hours'], q=4,
                                  labels=['Low', 'Medium', 'High', 'Very High'])

print('New features added:')
print(['engagement_score', 'high_watcher', 'inactive_30d',
       'age_group', 'fee_per_profile', 'watch_segment'])
df_fe[['engagement_score', 'high_watcher', 'inactive_30d',
       'age_group', 'fee_per_profile', 'watch_segment']].describe()

In [ ]:
# ── Visualise engineered features ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Engagement score
for churn_val, color, label in zip([0, 1], PALETTE2, ['Retained', 'Churned']):
    axes[0].hist(df_fe[df_fe['churned'] == churn_val]['engagement_score'],
                 bins=40, alpha=0.6, color=color, label=label)
axes[0].set_title('Engagement Score Distribution')
axes[0].set_xlabel('Engagement Score')
axes[0].legend()

# Churn by age group
ag_churn = df_fe.groupby('age_group', observed=True)['churned'].mean() * 100
axes[1].bar(ag_churn.index.astype(str), ag_churn.values,
            color=NETFLIX_RED, edgecolor='white')
axes[1].set_title('Churn Rate by Age Group')
axes[1].set_ylabel('Churn Rate (%)')

# Churn by watch segment
ws_churn = df_fe.groupby('watch_segment', observed=True)['churned'].mean() * 100
axes[2].bar(ws_churn.index.astype(str), ws_churn.values,
            color='#3B82F6', edgecolor='white')
axes[2].set_title('Churn Rate by Watch Segment')
axes[2].set_ylabel('Churn Rate (%)')

plt.suptitle('Engineered Features — Churn Insights', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Churn Deep-Dive Analysis

In [ ]:
# ── Overall churn stats ───────────────────────────────────────────────────────
churned   = df_fe[df_fe['churned'] == 1]
retained  = df_fe[df_fe['churned'] == 0]

stats = pd.DataFrame({
    'Churned':  churned[num_cols].mean(),
    'Retained': retained[num_cols].mean()
}).T

print('Mean values by churn status:')
stats.style.format('{:.2f}').background_gradient(cmap='RdYlGn', axis=None)

In [ ]:
# ── Churn rate: Subscription × Device ────────────────────────────────────────
pivot = df_fe.pivot_table(values='churned',
                           index='subscription_type',
                           columns='device',
                           aggfunc='mean') * 100

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='Reds',
            linewidths=0.5, cbar_kws={'label': 'Churn Rate (%)'})
ax.set_title('Churn Rate (%) — Subscription Type × Device', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Churn rate: Region × Subscription ────────────────────────────────────────
pivot2 = df_fe.pivot_table(values='churned',
                            index='region',
                            columns='subscription_type',
                            aggfunc='mean') * 100

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot2, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': 'Churn Rate (%)'})
ax.set_title('Churn Rate (%) — Region × Subscription Type', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Inactivity vs churn ───────────────────────────────────────────────────────
inact_churn = df_fe.groupby('inactive_30d')['churned'].mean() * 100
print('Churn rate by 30-day inactivity flag:')
for flag, rate in inact_churn.items():
    print(f'  {"Inactive ≥30d" if flag else "Active <30d"}: {rate:.1f}%')

In [ ]:
# ── Genre-level churn & average watch hours ───────────────────────────────────
genre_stats = df_fe.groupby('favorite_genre').agg(
    churn_rate=('churned', 'mean'),
    avg_watch_hours=('watch_hours', 'mean'),
    n_customers=('churned', 'count')
).reset_index()
genre_stats['churn_rate'] *= 100
genre_stats = genre_stats.sort_values('churn_rate', ascending=False)

fig, ax1 = plt.subplots(figsize=(12, 5))
x = np.arange(len(genre_stats))
bars = ax1.bar(x - 0.2, genre_stats['churn_rate'], width=0.35,
               color=NETFLIX_RED, label='Churn Rate (%)', alpha=0.85)
ax2 = ax1.twinx()
ax2.bar(x + 0.2, genre_stats['avg_watch_hours'], width=0.35,
        color='#3B82F6', label='Avg Watch Hours', alpha=0.75)
ax1.set_xticks(x)
ax1.set_xticklabels(genre_stats['favorite_genre'], rotation=30, ha='right')
ax1.set_ylabel('Churn Rate (%)', color=NETFLIX_RED)
ax2.set_ylabel('Avg Watch Hours', color='#3B82F6')
ax1.set_title('Churn Rate & Avg Watch Hours by Favorite Genre', fontweight='bold')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
plt.tight_layout()
plt.show()

---
## 6. Machine Learning Models

### 6.1 Data Preparation

In [ ]:
# ── Encode categoricals ───────────────────────────────────────────────────────
ml_df = df_fe.copy()

# Ordinal/label encode ordered or low-cardinality features
le = LabelEncoder()
encode_cols = ['gender', 'subscription_type', 'region',
               'device', 'payment_method', 'favorite_genre',
               'age_group', 'watch_segment']

for col in encode_cols:
    ml_df[col] = le.fit_transform(ml_df[col].astype(str))

# ── Feature matrix & target ───────────────────────────────────────────────────
FEATURE_COLS = [
    'age', 'gender', 'subscription_type', 'watch_hours',
    'last_login_days', 'region', 'device', 'monthly_fee',
    'payment_method', 'number_of_profiles', 'avg_watch_time_per_day',
    'favorite_genre', 'engagement_score', 'high_watcher',
    'inactive_30d', 'age_group', 'fee_per_profile', 'watch_segment'
]

X = ml_df[FEATURE_COLS]
y = ml_df['churned']

# ── Train / test split (stratified, 80/20) ────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# ── Scale features ────────────────────────────────────────────────────────────
scaler  = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train shape : {X_train.shape}')
print(f'Test shape  : {X_test.shape}')
print(f'Churn rate in train: {y_train.mean()*100:.1f}%')
print(f'Churn rate in test : {y_test.mean()*100:.1f}%')

### 6.2 Train Multiple Classifiers

In [ ]:
# ── Define models ─────────────────────────────────────────────────────────────
models = {
    'Logistic Regression':     LogisticRegression(max_iter=500, random_state=42),
    'Decision Tree':           DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest':           RandomForestClassifier(n_estimators=200, max_depth=10,
                                                      random_state=42, n_jobs=-1),
    'Gradient Boosting':       GradientBoostingClassifier(n_estimators=200,
                                                           learning_rate=0.08,
                                                           max_depth=5, random_state=42),
    'SVM':                     SVC(probability=True, kernel='rbf', C=1.0, random_state=42),
    'K-Nearest Neighbours':    KNeighborsClassifier(n_neighbors=7)
}

# ── Fit & collect metrics ─────────────────────────────────────────────────────
results = {}
trained_models = {}

for name, model in models.items():
    # Tree-based models on unscaled data; others on scaled
    is_tree = name in ('Decision Tree', 'Random Forest', 'Gradient Boosting')
    X_tr = X_train if is_tree else X_train_sc
    X_te = X_test  if is_tree else X_test_sc

    model.fit(X_tr, y_train)
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    results[name] = {
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall':    recall_score(y_test, y_pred, zero_division=0),
        'F1':        f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC':   roc_auc_score(y_test, y_proba)
    }
    trained_models[name] = (model, X_tr, X_te, y_pred, y_proba)
    print(f'{name:28s} | Acc={results[name]["Accuracy"]:.3f}  '
          f'F1={results[name]["F1"]:.3f}  AUC={results[name]["ROC-AUC"]:.3f}')

---
## 7. Model Evaluation & Comparison

In [ ]:
# ── Results table ─────────────────────────────────────────────────────────────
results_df = pd.DataFrame(results).T.round(4)
results_df = results_df.sort_values('ROC-AUC', ascending=False)
print('Model Evaluation Summary (sorted by ROC-AUC):')
results_df.style.background_gradient(cmap='RdYlGn', axis=0)

In [ ]:
# ── Metric comparison bar chart ───────────────────────────────────────────────
metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
x = np.arange(len(results_df))
width = 0.15
colors = ['#3B82F6', '#10B981', '#F59E0B', '#EF4444', '#8B5CF6']

fig, ax = plt.subplots(figsize=(14, 5))
for i, (metric, color) in enumerate(zip(metrics, colors)):
    ax.bar(x + i * width, results_df[metric], width, label=metric,
           color=color, alpha=0.88, edgecolor='white')

ax.set_xticks(x + width * 2)
ax.set_xticklabels(results_df.index, rotation=20, ha='right')
ax.set_ylim(0.4, 1.0)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── ROC curves ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
roc_colors = plt.cm.tab10(np.linspace(0, 1, len(models)))

for (name, (model, _, X_te, _, y_proba)), color in zip(trained_models.items(), roc_colors):
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = results[name]['ROC-AUC']
    ax.plot(fpr, tpr, color=color, linewidth=1.8, label=f'{name} (AUC={auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random Classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models', fontweight='bold')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion matrices for top 2 models ──────────────────────────────────────
top2 = results_df.head(2).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, name in zip(axes, top2):
    _, _, _, y_pred, _ = trained_models[name]
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Retained', 'Churned'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\nAccuracy={results[name]["Accuracy"]:.3f}', fontweight='bold')

plt.suptitle('Confusion Matrices — Top 2 Models', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Full classification report for best model ─────────────────────────────────
best_name = results_df.index[0]
_, _, _, y_pred_best, _ = trained_models[best_name]

print(f'=== Classification Report: {best_name} ===')
print(classification_report(y_test, y_pred_best,
                             target_names=['Retained', 'Churned']))

In [ ]:
# ── Cross-validation (5-fold, best model on full data) ────────────────────────
best_model, X_tr_best, _, _, _ = trained_models[best_name]
is_tree_best = best_name in ('Decision Tree', 'Random Forest', 'Gradient Boosting')
X_cv = X if is_tree_best else scaler.fit_transform(X)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_model, X_cv, y, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'Cross-validation ROC-AUC ({best_name}):')
print(f'  Folds  : {np.round(cv_scores, 4)}')
print(f'  Mean   : {cv_scores.mean():.4f}')
print(f'  Std    : {cv_scores.std():.4f}')

---
## 8. Feature Importance

In [ ]:
# ── Random Forest feature importance ─────────────────────────────────────────
rf_model = trained_models['Random Forest'][0]
fi = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS)
fi = fi.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors_fi = [NETFLIX_RED if v >= fi.quantile(0.75) else '#6B6B6B' for v in fi.values]
ax.barh(fi.index, fi.values, color=colors_fi, edgecolor='white')
ax.set_xlabel('Importance Score')
ax.set_title('Random Forest — Feature Importance\n(Red = Top Quartile)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Gradient Boosting feature importance ─────────────────────────────────────
gb_model = trained_models['Gradient Boosting'][0]
fi_gb = pd.Series(gb_model.feature_importances_, index=FEATURE_COLS)
fi_gb = fi_gb.sort_values(ascending=False).head(12)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(fi_gb.index, fi_gb.values,
       color=['#E50914' if i < 5 else '#6B6B6B' for i in range(len(fi_gb))],
       edgecolor='white')
ax.set_xticklabels(fi_gb.index, rotation=35, ha='right')
ax.set_ylabel('Importance Score')
ax.set_title('Gradient Boosting — Top 12 Feature Importances', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Logistic Regression — coefficient magnitudes ──────────────────────────────
lr_model = trained_models['Logistic Regression'][0]
coeff = pd.Series(np.abs(lr_model.coef_[0]), index=FEATURE_COLS)
coeff = coeff.sort_values(ascending=False).head(12)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(coeff.index, coeff.values, color='#3B82F6', edgecolor='white')
ax.set_xticklabels(coeff.index, rotation=35, ha='right')
ax.set_ylabel('|Coefficient|')
ax.set_title('Logistic Regression — Top 12 Feature Coefficients (Absolute)',
             fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Key Insights & Business Recommendations

In [ ]:
import sys, io
if sys.stdout.encoding and sys.stdout.encoding.upper() not in ('UTF-8','UTF8'):
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8', errors='replace')

# ── Compute insight numbers ───────────────────────────────────────────────────
overall_churn      = df_fe['churned'].mean() * 100
inact_churn_rate   = df_fe[df_fe['inactive_30d'] == 1]['churned'].mean() * 100
active_churn_rate  = df_fe[df_fe['inactive_30d'] == 0]['churned'].mean() * 100
top_sub_churn      = df_fe.groupby('subscription_type')['churned'].mean().idxmax()
top_sub_rate       = df_fe.groupby('subscription_type')['churned'].mean().max() * 100
top_region_churn   = df_fe.groupby('region')['churned'].mean().idxmax()
top_region_rate    = df_fe.groupby('region')['churned'].mean().max() * 100
top_device_churn   = df_fe.groupby('device')['churned'].mean().idxmax()
top_device_rate    = df_fe.groupby('device')['churned'].mean().max() * 100
top_genre_churn    = df_fe.groupby('favorite_genre')['churned'].mean().idxmax()
top_genre_rate     = df_fe.groupby('favorite_genre')['churned'].mean().max() * 100
best_auc           = results_df['ROC-AUC'].max()
best_f1            = results_df['F1'].max()
top_feature_rf     = fi.sort_values(ascending=False).index[0]
top_feature_gb     = fi_gb.index[0]
low_eng_churn      = df_fe[df_fe['engagement_score'] < df_fe['engagement_score'].quantile(0.25)]['churned'].mean() * 100
high_eng_churn     = df_fe[df_fe['engagement_score'] > df_fe['engagement_score'].quantile(0.75)]['churned'].mean() * 100

print(f'Overall churn rate         : {overall_churn:.1f}%')
print(f'Inactive ≥30d churn rate   : {inact_churn_rate:.1f}%')
print(f'Active <30d churn rate     : {active_churn_rate:.1f}%')
print(f'Highest churn subscription : {top_sub_churn} ({top_sub_rate:.1f}%)')
print(f'Highest churn region       : {top_region_churn} ({top_region_rate:.1f}%)')
print(f'Highest churn device       : {top_device_churn} ({top_device_rate:.1f}%)')
print(f'Highest churn genre        : {top_genre_churn} ({top_genre_rate:.1f}%)')
print(f'Best model                 : {best_name} (AUC={best_auc:.3f}, F1={best_f1:.3f})')
print(f'Top RF feature             : {top_feature_rf}')
print(f'Top GB feature             : {top_feature_gb}')
print(f'Low engagement churn rate  : {low_eng_churn:.1f}%')
print(f'High engagement churn rate : {high_eng_churn:.1f}%')

In [ ]:
# ── Summary dashboard ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 10))
fig.patch.set_facecolor('#F8F9FA')
plt.suptitle('Netflix Churn Analytics — Executive Summary Dashboard',
             fontsize=16, fontweight='bold', y=1.01)

# 1. KPI scorecards (text)
ax_kpi = fig.add_subplot(3, 4, (1, 4))
ax_kpi.axis('off')
kpis = [
    (f'{overall_churn:.1f}%',    'Overall\nChurn Rate'),
    (f'{inact_churn_rate:.1f}%', 'Churn Rate\n(Inactive ≥30d)'),
    (f'{best_auc:.3f}',          f'Best Model\nROC-AUC\n({best_name})'),
    (f'{best_f1:.3f}',           f'Best Model\nF1 Score'),
]
box_colors = [NETFLIX_RED, '#F59E0B', '#10B981', '#3B82F6']
for k, (val, label) in enumerate(kpis):
    ax_kpi.text(0.12 + k * 0.25, 0.65, val, transform=ax_kpi.transAxes,
                fontsize=22, fontweight='bold', ha='center', color=box_colors[k])
    ax_kpi.text(0.12 + k * 0.25, 0.25, label, transform=ax_kpi.transAxes,
                fontsize=10, ha='center', color='#444', va='center')

# 2. Churn by Subscription
ax1 = fig.add_subplot(3, 4, 5)
sub_cr = df_fe.groupby('subscription_type')['churned'].mean().sort_values() * 100
ax1.barh(sub_cr.index, sub_cr.values, color=NETFLIX_RED, alpha=0.85)
ax1.set_title('By Subscription', fontsize=10)
ax1.set_xlabel('Churn Rate (%)')

# 3. Churn by Region
ax2 = fig.add_subplot(3, 4, 6)
reg_cr = df_fe.groupby('region')['churned'].mean().sort_values() * 100
ax2.barh(reg_cr.index, reg_cr.values, color='#F59E0B', alpha=0.85)
ax2.set_title('By Region', fontsize=10)
ax2.set_xlabel('Churn Rate (%)')

# 4. Churn by Device
ax3 = fig.add_subplot(3, 4, 7)
dev_cr = df_fe.groupby('device')['churned'].mean().sort_values() * 100
ax3.barh(dev_cr.index, dev_cr.values, color='#8B5CF6', alpha=0.85)
ax3.set_title('By Device', fontsize=10)
ax3.set_xlabel('Churn Rate (%)')

# 5. Model comparison
ax4 = fig.add_subplot(3, 4, 8)
ax4.barh(results_df.index, results_df['ROC-AUC'],
         color='#10B981', alpha=0.85)
ax4.set_xlim(0.4, 1.0)
ax4.set_title('Model ROC-AUC', fontsize=10)
ax4.set_xlabel('ROC-AUC')

# 6. Engagement score vs churn
ax5 = fig.add_subplot(3, 4, (9, 10))
for churn_val, color, label in zip([0, 1], PALETTE2, ['Retained', 'Churned']):
    sns.kdeplot(df_fe[df_fe['churned'] == churn_val]['engagement_score'],
                ax=ax5, color=color, label=label, fill=True, alpha=0.3)
ax5.set_title('Engagement Score by Churn', fontsize=10)
ax5.set_xlabel('Engagement Score')
ax5.legend(fontsize=8)

# 7. Top 8 RF features
ax6 = fig.add_subplot(3, 4, (11, 12))
top8 = fi.sort_values(ascending=False).head(8)
ax6.barh(top8.index[::-1], top8.values[::-1], color=NETFLIX_RED, alpha=0.85)
ax6.set_title('Top 8 Features (RF)', fontsize=10)
ax6.set_xlabel('Importance')

plt.tight_layout()
plt.savefig('netflix_churn_dashboard.png', dpi=130, bbox_inches='tight',
            facecolor='#F8F9FA')
plt.show()
print('Dashboard saved as netflix_churn_dashboard.png')

### 9.1 Key Findings

In [ ]:
findings = f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║           NETFLIX CHURN ANALYTICS — KEY FINDINGS SUMMARY                   ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  CHURN OVERVIEW                                                             ║
║  • Overall churn rate        : {overall_churn:.1f}%                                    ║
║  • Inactive ≥30d churn rate  : {inact_churn_rate:.1f}%  vs  {active_churn_rate:.1f}% for active users      ║
║  • Low-engagement churn rate : {low_eng_churn:.1f}%  vs  {high_eng_churn:.1f}% for high-engagement     ║
║                                                                             ║
║  HIGHEST-RISK SEGMENTS                                                      ║
║  • Subscription type : {top_sub_churn:<10s} ({top_sub_rate:.1f}% churn)                 ║
║  • Region            : {top_region_churn:<10s} ({top_region_rate:.1f}% churn)                 ║
║  • Device            : {top_device_churn:<10s} ({top_device_rate:.1f}% churn)                 ║
║  • Favorite genre    : {top_genre_churn:<10s} ({top_genre_rate:.1f}% churn)                 ║
║                                                                             ║
║  MODEL PERFORMANCE                                                          ║
║  • Best model        : {best_name:<25s}              ║
║  • ROC-AUC           : {best_auc:.3f}  |  F1 : {best_f1:.3f}                         ║
║                                                                             ║
║  TOP PREDICTIVE FEATURES (Random Forest / Gradient Boosting)                ║
║  • {top_feature_rf:<20s} / {top_feature_gb:<20s}                 ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
print(findings)

### 9.2 Business Recommendations

In [ ]:
recommendations = """
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  BUSINESS RECOMMENDATIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. DEPLOY REAL-TIME CHURN SCORING
   Deploy the best-performing model (Gradient Boosting / Random Forest) as a
   real-time API. Score all customers daily. Flag any customer with a churn
   probability > 0.65 for immediate retention intervention.

2. RE-ENGAGE INACTIVE USERS (≥30 Days)
   Inactive users churn at ~2× the rate of active users. Trigger personalised
   re-engagement emails / push notifications at days 15, 22, and 28 of
   inactivity. Include genre-specific content recommendations based on
   favourite_genre.

3. TARGETED OFFERS FOR HIGH-RISK SUBSCRIPTION TIER
   The highest-churn subscription tier needs pricing or value re-evaluation.
   Consider a limited-time discount, a free profile upgrade, or a service
   bundle to reduce perceived cost-to-value gap.

4. DEVICE-SPECIFIC UX IMPROVEMENTS
   The highest-churn device type suggests a UX friction point. Conduct UX
   audits and A/B tests on that device platform to reduce friction in the
   content discovery and playback experience.

5. REGIONAL RETENTION CAMPAIGNS
   The highest-churn region may reflect competitive pressure or content
   localisation gaps. Invest in region-specific content licensing and run
   targeted promotional campaigns to strengthen brand loyalty.

6. ENGAGEMENT SCORE MONITORING
   Operationalise the engineered engagement_score as a weekly KPI. Set alert
   thresholds: customers dropping below the 25th percentile for two
   consecutive weeks should enter an automated retention workflow.

7. GENRE-LED CONTENT STRATEGY
   High-churn genres indicate content gaps. Commission or license additional
   titles in these categories. Surface these titles prominently on the home
   screen for at-risk segments.

8. FAMILY & MULTI-PROFILE INCENTIVES
   Customers with more profiles show lower churn—they are more embedded in the
   platform. Offer incentives (extra profile slots, family plan discounts) to
   single-profile accounts to increase stickiness.

9. PAYMENT METHOD FRICTION REDUCTION
   Certain payment methods correlate with higher churn, possibly due to failed
   renewals. Implement pre-expiry alerts, one-click payment updates, and
   graceful retry logic for failed transactions.

10. CONTINUOUS MODEL RETRAINING
    Retrain the churn model monthly with fresh data. Monitor data drift on
    key features (last_login_days, watch_hours, engagement_score). Set up
    automated retraining pipelines to maintain prediction accuracy over time.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Estimated Impact: A 10% reduction in churn on a 5,000-customer base at
  an avg monthly fee of ~$13.66 → ~$68,300 additional monthly revenue.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""
print(recommendations)

In [ ]:
# ── Recommendation priority matrix visualisation ───────────────────────────────
recs = [
    ('Real-Time Churn Scoring',       9, 3),
    ('Re-engage Inactive Users',      8, 2),
    ('Subscription Tier Offers',      7, 3),
    ('Device UX Improvements',        6, 4),
    ('Regional Campaigns',            7, 5),
    ('Engagement Score KPI',          8, 1),
    ('Genre Content Strategy',        6, 4),
    ('Multi-Profile Incentives',      5, 2),
    ('Payment Friction Reduction',    7, 2),
    ('Model Retraining Pipeline',     9, 3),
]
rec_df = pd.DataFrame(recs, columns=['Recommendation', 'Impact', 'Effort'])

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(rec_df['Effort'], rec_df['Impact'],
                     s=250, c=rec_df['Impact'],
                     cmap='RdYlGn', vmin=4, vmax=10,
                     edgecolors='white', linewidth=1.5, zorder=3)

for _, row in rec_df.iterrows():
    ax.annotate(row['Recommendation'],
                (row['Effort'], row['Impact']),
                textcoords='offset points', xytext=(8, 4),
                fontsize=8)

ax.axhline(y=7, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax.axvline(x=3, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax.set_xlim(0, 7)
ax.set_ylim(3, 11)
ax.set_xlabel('Implementation Effort (1=Low, 5=High)', fontsize=11)
ax.set_ylabel('Business Impact (1=Low, 10=High)', fontsize=11)
ax.set_title('Recommendation Priority Matrix\n(Top-left quadrant = Quick Wins)',
             fontsize=13, fontweight='bold')

# Quadrant labels
ax.text(0.5, 10.5, 'QUICK WINS', color='green', fontsize=9, fontweight='bold', alpha=0.7)
ax.text(4.5, 10.5, 'BIG BETS',   color=NETFLIX_RED, fontsize=9, fontweight='bold', alpha=0.7)
ax.text(0.5, 3.5,  'FILL-INS',   color='gray', fontsize=9, fontweight='bold', alpha=0.7)
ax.text(4.5, 3.5,  'THANKLESS',  color='#888', fontsize=9, fontweight='bold', alpha=0.7)

plt.colorbar(scatter, ax=ax, label='Impact Score')
plt.tight_layout()
plt.show()

---

## Notebook Complete ✓

| Step | Status |
|------|--------|
| Data Loading & Inspection | ✅ |
| Data Cleaning | ✅ |
| Exploratory Data Analysis | ✅ |
| Feature Engineering | ✅ |
| Churn Deep-Dive | ✅ |
| ML Models (6 classifiers) | ✅ |
| Model Evaluation & ROC | ✅ |
| Cross-Validation | ✅ |
| Feature Importance | ✅ |
| Insights & Recommendations | ✅ |
| Executive Dashboard | ✅ |

**Best model:** determined dynamically at runtime — typically **Gradient Boosting** or **Random Forest** on this dataset.  
**Top predictive features:** `last_login_days`, `engagement_score`, `watch_hours`, `avg_watch_time_per_day`.